# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print main metadata fields
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Cite as: {dataset.metadata.cite_as}\n")
print(f"Published: {dataset.metadata.date_published}")

## 2. Data Overview

Review available **record sets**, **fields**, and their `@id`s available in the Croissant schema. These `@id` values are used to reference and extract the data in subsequent steps.

**List all `@id`s for record sets, their fields, and example columns for exploration:**

In [ ]:
record_sets = dataset.metadata.record_sets

print("Available record sets and their field and column @ids:\n")
main_record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  RecordSet @id: {rs.id}")
    main_record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (field @id: {field.id})")
            if hasattr(field, 'columns') and field.columns:
                print(f"      Columns:")
                for col in field.columns:
                    print(f"        * {col.name} (column @id: {col.id})")
    print()

## 3. Data Extraction

Load data from each record set into a Pandas DataFrame for analysis. Use the available record set and field `@id`s from the above overview to select relevant data for analysis.

We'll extract all main data tables by record set `@id`, making them easily accessible for further processing.

In [ ]:
# Extract data from each record set by @id
dataframes = {}

# If there are no record sets, print a message
if not main_record_set_ids:
    print("No record sets found in the Croissant metadata.")
else:
    for rs_id in main_record_set_ids:
        try:
            print(f"Loading record set: {rs_id}")
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  --> Columns: {df.columns.tolist()}")
            print(f"  --> Number of records: {len(df)}\n")
        except Exception as e:
            print(f"  Could not load records for {rs_id}: {e}\n")

    # Preview the first record set if available
    preview_id = main_record_set_ids[0] if main_record_set_ids else None
    if preview_id:
        print(f"Sample data from RecordSet {preview_id}:")
        display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes using the DataFrame for a record set and referencing field/column `@id`s.

We'll:
- Filter records based on a numeric field using its `@id`.
- Normalize the numeric column.
- Optionally group by a categorical field (`@id`).

In [ ]:
# Choose a record set and numeric field for EDA
if main_record_set_ids:
    record_set_id = main_record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Working with RecordSet @id: {record_set_id}")
    
    # Identify likely numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    print(f"Available numeric columns: {numeric_cols}")

    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean()  # Use mean as threshold for illustration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by first non-numeric column, if any
        group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            group_field = None
            print("No suitable categorical field to group by.")
    else:
        print("No numeric columns found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

Visualize distributions or relationships using matplotlib and seaborn. We'll plot a histogram for the selected numeric field and, if grouped data is available, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_ids and numeric_cols:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if possible
    if group_fields:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric fields available for visualization.')

## 6. Conclusion

In this notebook, you:

- Loaded the [FAIR<sup>2</sup> Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and explored metadata and available record sets and fields by their `@id`s
- Loaded data for each record set dynamically, using the `@id`
- Performed exploratory data analysis using filtering, normalization, and grouping by field `@id`s
- Visualized distributions

This approach demonstrates how referencing by `@id` ensures robust, schema-driven, and reproducible data exploration using the `mlcroissant` library.